# 21｜葡萄牙污水 DDA 原始数据解包与样品清单冻结

本笔记本是真实水样案例的第一步：校验 Zenodo 原始压缩包，只解压 `AutoMSMS_DDA`，冻结 96 个 Agilent `.d` 数据集及后续 mzML 转换计划。

本步不转换 mzML、不做峰提取、不做 PBT 预测，也不解压 DIA；这些数据与 ChemVista 外部盲测严格分开。

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd
from IPython.display import display

ROOT = Path(r'E:\\谱图预测PBT_终版')
ENV_MS = Path(r'D:\\Appinstallation\\Anaconda\\envs\\env_ms\\python.exe')
SRC = ROOT / 'src'
CASE = ROOT / 'external_data/portugal_wastewater_5830725'
ARCHIVE = CASE / '00_archive/Portugal_wastewater_LCHRMS_dataset.zip'
MANIFEST_DIR = CASE / 'manifests'

# True = 校验后解压约 5.24 GB DDA；False = 只做档案审计，不解压。
RUN_DDA_EXTRACTION = True
# 正常保持 False；重跑时会自动跳过已完整文件。
REEXTRACT_EXISTING = False

assert ENV_MS.is_file(), ENV_MS
assert ARCHIVE.is_file(), ARCHIVE
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
disk = shutil.disk_usage(CASE)
print('ROOT:', ROOT)
print('Notebook kernel:', sys.executable)
print('env_ms:', ENV_MS)
print(f'Archive: {ARCHIVE.stat().st_size / 1024**3:.2f} GiB')
print(f'E drive free: {disk.free / 1024**3:.1f} GiB')

## 1. 前置门控

20 号必须已完成且全部 QC 通过。本单元只读取其最终清单，不会重跑 ChemVista。

In [ ]:
nb20_path = ROOT / 'outputs/pbt_msms_v1/chemvista_external_20_v1/20_run_manifest.json'
nb20 = json.loads(nb20_path.read_text(encoding='utf-8'))
assert nb20.get('status') == 'READY_FOR_NOTEBOOK_21', nb20.get('status')
assert nb20.get('qc_all_pass') is True
print('✓ 20号已解锁：', nb20['status'])

## 2. 校验原始包并解压 DDA

下面只调用一次主程序，避免重复计算 16.94 GiB 压缩包的 SHA-256。默认解压正、负离子模式的 96 个 `AutoMSMS_DDA` `.d` 目录，不解压 `All_Ions_DIA`。

运行时先长时间无输出属于正常现象：此时正在逐块计算整个压缩包的 SHA-256。解压阶段会显示进度条；中断后可直接重跑，已完整文件会跳过。

In [ ]:
stage = 'extract' if RUN_DDA_EXTRACTION else 'audit'
cmd = [str(ENV_MS), '-m', 'pbt_msms.portugal_wastewater',
       '--stage', stage, '--root', str(ROOT)]
if REEXTRACT_EXISTING:
    cmd.append('--reextract')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=ROOT, env=env)
print()
print('✓ Notebook 21 主程序执行完成')

## 3. 结果与 QC

完整解压后必须显示 `READY_FOR_NOTEBOOK_22`。公开包中没有能由文件名明确识别的 blank/QC，这是案例限制，不影响解压通过，但后文不能声称已做原始空白扣除。`A/B` 是两次迭代 DDA 进样，不是生物学重复。

In [ ]:
run_manifest_path = MANIFEST_DIR / '21_portugal_dda_extraction_manifest.json'
dataset_manifest_path = MANIFEST_DIR / '21_portugal_dda_dataset_manifest.csv'
qc_path = MANIFEST_DIR / '21_portugal_dda_qc_checks.csv'
conversion_plan_path = MANIFEST_DIR / '21_portugal_msconvert_plan.csv'

run_manifest = json.loads(run_manifest_path.read_text(encoding='utf-8'))
datasets = pd.read_csv(dataset_manifest_path)
qc = pd.read_csv(qc_path)

display(pd.Series({
    'status': run_manifest['status'],
    'stage': run_manifest['stage'],
    'DDA datasets': run_manifest['counts']['dda_datasets'],
    'DDA size (GiB)': round(run_manifest['archive']['dda_uncompressed_bytes'] / 1024**3, 3),
    'DIA retained in ZIP (GiB)': round(run_manifest['archive']['dia_uncompressed_bytes'] / 1024**3, 3),
    'new files extracted': run_manifest['extraction']['extracted_files'],
    'existing files skipped': run_manifest['extraction']['skipped_existing_files'],
}, name='value').to_frame())
display(pd.crosstab([datasets['site_code'], datasets['campaign']],
                    [datasets['mode'], datasets['injection']]))
display(qc)

expected = 'READY_FOR_NOTEBOOK_22' if RUN_DDA_EXTRACTION else 'READY_FOR_DDA_EXTRACTION'
assert run_manifest['status'] == expected, run_manifest['status']
assert len(datasets) == 96
assert conversion_plan_path.is_file()
print()
print('✓ 最终状态:', run_manifest['status'])
print('下一步：22号使用 ProteoWizard MSConvert 做厂商峰提取并转换 mzML。')